# Proxy Splenial Angle (p-SA) — MRI Adaptation

Three-step pipeline:
1. **Sagittal**: Segment 3rd ventricle on mid-sagittal plane to find starting axial plane
2. **Axial**: Determine optimal axial plane via white matter profile in corpus callosum
3. **Axial**: Segment occipital horns of lateral ventricles and fit lines to measure angle

**MRI adaptation**: No HU windowing. For OASIS, sagittal slices via `arr[:, :, x]` and axial via `arr[z, :, :]`. For ABCD, use `extract_sagittal_view()`.

**Reference**: Samadani et al., JNS 2024

## Library imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import statsmodels.api as sm
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, getLargestCC, elliptical_mask, fit_lines,
                       extract_sagittal_view, normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL,
                       parse_acpc_coords, load_acpc_from_txt)

## Configuration

Set `DATASET` to match your data source. The notebook adapts file discovery and PC coordinate loading accordingly.

Supported datasets:
- **`"oasis1"`** — OASIS-1 cross-sectional (skull-stripped, atlas-registered `.img` files). AC/PC in T88 voxel space, hardcoded from `mri_utils`.
- **`"abcd"`** — ABCD Study T1w scans (ART-corrected `.nii`/`.nii.gz` files). AC/PC loaded per-subject from `.txt` sidecar files.
- **`"custom"`** — Any NIfTI dataset; set `SCAN_GLOB` and provide AC/PC loading logic.

**Note:** p-SA uses both sagittal and axial views. For OASIS (isotropic, atlas-registered), the sagittal plane is extracted as `arr[:, :, x]` directly from the 3D array. For ABCD, `extract_sagittal_view()` permutes the SimpleITK volume axes before converting to array.

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1
    # AC/PC are fixed for OASIS T88 atlas space (from mri_utils)
    # p-SA uses only PC for finding the sagittal slice; OASIS_PC_VOXEL = (85, 87, 73)
    # For OASIS, sagittal extraction: arr[:, :, pc_x] where pc_x = OASIS_PC_VOXEL[0]
    ACPC_MODE = "oasis_fixed"  # all subjects share the same atlas-space coordinates

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None  # set path to ABCD demographics CSV/TSV if available
    # ART-corrected T1w scans (output from Charlie's ART batch script)
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX
    # For ABCD, AC/PC are loaded per-subject from sidecar .txt files
    ACPC_MODE = "abcd_txt"  # per-subject .txt sidecar

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
    ACPC_MODE = "custom"  # implement get_pc_voxel() below for your dataset

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset: {DATASET}")
print(f"Data dir: {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")
print(f"AC/PC mode: {ACPC_MODE}")

## Processing parameters

Three independent parameter dicts, one per pipeline step.

**Step 1 (`start_ax_plane_processing_params`):** Controls 3rd ventricle segmentation on the mid-sagittal plane. The tilted elliptical ROI (`third_ven_roi_tilt_degrees`) accounts for the anatomical tilt of the 3V relative to the axial plane.

**Step 2 (`opt_ax_plane_processing_params`):** Controls the search for the optimal axial plane. Starting from the `start_ax_plane`, the algorithm searches superiorly (up to `max_superior_search_ax_planes` slices) for the plane with the richest white matter signal between the lateral ventricles.

**Step 3 (`occipital_horns_segment_params`):** Controls occipital horn segmentation. The `ven_seg_to_full_brain_ratio` threshold removes non-ventricle dark regions that are too large (e.g., sulci).

**MRI note:** HU window params (`brain_window_center`, `brain_window_width`, `csf_window_center`, `csf_window_width`) are entirely absent — they are not needed for MRI. Otsu thresholding replaces all HU-based intensity gating.

In [ ]:
# Step 1: 3V segmentation on MSP to find starting axial plane
start_ax_plane_processing_params = {
    "third_ven_seg_initial_gauss_blur_size": 3,
    "msp_adaptive_thresh_size": 75,
    "third_ven_cen_roi_row_offset": 10,
    "third_ven_cen_roi_col_offset": 10,
    "third_ven_roi_major_axis": 35,
    "third_ven_roi_minor_axis": 15,
    "third_ven_roi_tilt_degrees": -75,
}

# Step 2: Optimal axial plane via WM profile
opt_ax_plane_processing_params = {
    "ax_adaptive_thresh_size": 75,
    "ax_lat_ven_roi_semi_major_axis": 50,
    "ax_lat_ven_roi_semi_minor_axis": 40,
    "lv_roi_close_kernel_size": 6,
    "num_close_iters_lv_roi_ax": 5,
    "max_superior_search_ax_planes": 15,
    "cc_roi_col_offset": 10,
    "ax_plane_gauss_blur_kernel_size": 3,
}

# Step 3: Occipital horn segmentation
occipital_horns_segment_params = {
    "lat_ven_seg_initial_gauss_blur_size": 3,
    "lv_roi_close_kernel_size": 6,
    "num_close_iters_roi_ax": 5,
    "ax_adaptive_thresh_size": 75,
    "ven_seg_to_full_brain_ratio": 0.1,
    "bilateral_filter_size": 11,
    "bilater_filter_sigma_color": 75,
    "bilater_filter_sigma_space": 75,
    "lv_middle_roi_col_offset": 5,
    "lv_middle_roi_row_offset": 20,
    "close_roi_left_prcnt_fbc": 0.2,
    "close_roi_cen_exclude": 0.1,
    "ax_lv_close_iter": 3,
    "ven_median_smooth_factor": 7,
    "min_ven_area_thresh": 20,
}

print("Processing parameters loaded.")
print(f"  Step 1 params: {list(start_ax_plane_processing_params.keys())}")
print(f"  Step 2 params: {list(opt_ax_plane_processing_params.keys())}")
print(f"  Step 3 params: {list(occipital_horns_segment_params.keys())}")

## Step 1: Determine starting axial plane via 3V segmentation

The posterior commissure (PC) is a reliable anatomical anchor on the mid-sagittal plane. The 3rd ventricle lies superior and anterior to the PC. By segmenting the 3V on the MSP, we identify the superior boundary of the 3V — which becomes our `start_ax_plane` for the subsequent axial search.

**MSP extraction strategy:**
- **OASIS** (atlas-registered, isotropic 1mm): the brain is already in T88 space, so the mid-sagittal plane is at a known x-index. We extract it as `arr[:, :, pc_x]` — a direct numpy slice from the 3D axial array.
- **ABCD** (subject-native space): `extract_sagittal_view()` permutes the SimpleITK volume axes (`PermuteAxes([1, 2, 0])`) so the x-axis becomes the depth dimension, then we convert to array and index by the AC/PC x-voxel from the sidecar file.

**MRI adaptation:** No HU windowing. The 3V appears as a dark (CSF) midline structure. Gaussian blur followed by Otsu + adaptive threshold isolates it from surrounding brain tissue.

In [ ]:
def determine_start_ax_plane(ax_img_arr, pc_voxel, params, visualize=False):
    """Segments 3V on mid-sagittal plane to find starting axial plane for p-SA.

    For MRI: operates on raw intensity array. No windowing.
    Sagittal MSP extracted via arr[:, :, pc_x] (OASIS) or from permuted volume (ABCD).

    Parameters
    ----------
    ax_img_arr : np.ndarray
        3D array in axial orientation, shape (Z, Y, X).
    pc_voxel : tuple of float
        Posterior commissure voxel coordinates (x, y, z) in the axial volume.
    params : dict
        Processing parameters from ``start_ax_plane_processing_params``.
    visualize : bool
        If True, plot MSP and segmentation for debugging.

    Returns
    -------
    start_ax_plane : int
        Axial slice index at the superior boundary of the 3rd ventricle.
    msp_image : np.ndarray
        2D mid-sagittal plane image used for 3V segmentation.
    seg_area : np.ndarray
        Binary mask of the segmented 3V region on the MSP.
    mcs : int
        Row index of the 3V centroid on the MSP (used downstream in Step 2).
    mrs : int
        Column index of the 3V centroid on the MSP (used downstream in Step 2).
    """
    pc_x = int(round(pc_voxel[0]))
    pc_y = int(round(pc_voxel[1]))
    pc_z = int(round(pc_voxel[2]))

    # Extract mid-sagittal plane from axial volume
    # arr shape is (Z, Y, X) so sagittal at x=pc_x is arr[:, :, pc_x]
    # rows = Z (superior is lower index), cols = Y (anterior is lower index)
    msp_image = ax_img_arr[:, :, pc_x].astype(np.float32)

    # Gaussian blur to suppress noise before thresholding
    blur_k = params["third_ven_seg_initial_gauss_blur_size"]
    msp_blurred = cv.GaussianBlur(msp_image, (blur_k, blur_k), 0)

    # Normalize to uint8 for OpenCV adaptive threshold
    msp_norm = cv.normalize(msp_blurred, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

    # Otsu on full MSP to get a global foreground mask
    try:
        otsu_thresh = filters.threshold_otsu(msp_norm)
    except ValueError:
        raise RuntimeError("Otsu threshold failed on MSP — check volume is non-empty.")
    msp_otsu = (msp_norm > otsu_thresh).astype(np.uint8)

    # Adaptive threshold for local contrast (captures 3V as a dark region)
    adapt_size = params["msp_adaptive_thresh_size"]
    if adapt_size % 2 == 0:
        adapt_size += 1
    msp_adapt = cv.adaptiveThreshold(
        msp_norm, 255,
        cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY_INV,
        adapt_size, 2
    )

    # Combined: must be bright in Otsu AND locally dark in adaptive
    # This isolates internal dark regions (ventricles) vs external dark (background)
    msp_combined = np.uint8(msp_otsu) * np.uint8(msp_adapt > 0)

    # Define tilted elliptical ROI centred slightly superior/anterior to PC
    # PC voxel in MSP coords: row = pc_z (axial depth), col = pc_y (anterior)
    cen_row = pc_z - params["third_ven_cen_roi_row_offset"]
    cen_col = pc_y + params["third_ven_cen_roi_col_offset"]
    roi_a = params["third_ven_roi_major_axis"]
    roi_b = params["third_ven_roi_minor_axis"]
    tilt_rad = np.radians(params["third_ven_roi_tilt_degrees"])

    roi_mask = elliptical_mask(
        cen=(cen_row, cen_col),
        a=roi_a, b=roi_b,
        shape=msp_combined.shape,
        angle=tilt_rad
    )

    # Apply ROI mask
    seg_area = msp_combined * roi_mask.astype(np.uint8)

    # Find connected components in ROI, keep largest (= 3V)
    labeled = measure.label(seg_area)
    if labeled.max() == 0:
        raise RuntimeError(
            "No connected components found in 3V ROI on MSP. "
            "Check pc_voxel, ROI offsets, and image orientation."
        )
    seg_area = getLargestCC(labeled).astype(np.uint8)

    # Centroid of 3V on MSP
    props = measure.regionprops(seg_area)
    if not props:
        raise RuntimeError("No region properties found after largest-CC selection.")
    centroid = props[0].centroid  # (row, col) on MSP = (z, y) in 3D
    mcs = int(round(centroid[0]))  # row on MSP = axial slice index
    mrs = int(round(centroid[1]))  # col on MSP = coronal position

    # Superior boundary of 3V on MSP = lowest axial index (rows decrease superiorly)
    ven_rows = np.where(seg_area > 0)[0]
    start_ax_plane = int(ven_rows.min())  # most superior row = smallest index

    if visualize:
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))
        axes[0].set_title("Mid-sagittal plane (raw)")
        axes[0].imshow(msp_image, cmap="gray")
        axes[0].axhline(pc_z, color="cyan", linestyle="--", linewidth=1, label="PC row")
        axes[0].legend(fontsize=8)

        axes[1].set_title("Otsu + adaptive combined")
        axes[1].imshow(msp_combined, cmap="gray")

        axes[2].set_title("Tilted elliptical ROI")
        axes[2].imshow(msp_image, cmap="gray")
        roi_vis = np.ma.masked_where(roi_mask == 0, roi_mask)
        axes[2].imshow(roi_vis, cmap="autumn", alpha=0.4)

        axes[3].set_title(f"3V segmentation (start ax plane={start_ax_plane})")
        axes[3].imshow(msp_image, cmap="gray")
        seg_vis = np.ma.masked_where(seg_area == 0, seg_area)
        axes[3].imshow(seg_vis, cmap="Greens", alpha=0.6)
        axes[3].axhline(start_ax_plane, color="red", linewidth=1.5, label=f"start_ax={start_ax_plane}")
        axes[3].plot(mrs, mcs, "r+", markersize=10)
        axes[3].legend(fontsize=8)

        plt.tight_layout()
        plt.show()

    return start_ax_plane, msp_image, seg_area, mcs, mrs

## Step 2: Determine optimal axial plane via WM profile

The proxy splenial angle must be measured on the specific axial plane where the corpus callosum (CC) splenium is most visible — typically slightly superior to the 3V superior boundary found in Step 1.

**Algorithm:** Starting at `start_ax_plane`, iterate superiorly through up to `max_superior_search_ax_planes` axial slices. On each slice:
1. Segment lateral ventricles (Otsu + adaptive + close)
2. Define a narrow CC ROI in the white matter strip between the two lateral ventricle bodies
3. Count supra-threshold (WM) pixels in the CC ROI

The optimal plane is the one with the **peak WM count** in the CC ROI — this corresponds to the slice where the CC splenium is thickest and most clearly delineated from adjacent CSF.

**MRI adaptation:** CT used a fixed HU range (22–28 HU) to identify white matter. For MRI, there are no standardized intensity units. Instead, we apply Otsu threshold on each axial slice and count pixels **above** the threshold in the CC ROI. On T1w MRI, WM is brighter than GM, which is brighter than CSF, so supra-Otsu pixels in the CC strip reliably represent WM.

In [ ]:
def determine_opt_ax_plane(start_ax_plane, ax_img_arr, msp_image, seg_area, mcs, mrs, params, visualize=False):
    """Finds optimal axial plane by analyzing WM profile in corpus callosum.

    Iterates superiorly from ``start_ax_plane``, quantifies white matter signal
    in the CC ROI on each slice, and returns the plane with the peak WM count.

    For MRI: uses Otsu threshold per slice instead of HU range for WM detection.

    Parameters
    ----------
    start_ax_plane : int
        Starting axial slice index from Step 1.
    ax_img_arr : np.ndarray
        3D array in axial orientation, shape (Z, Y, X).
    msp_image : np.ndarray
        2D mid-sagittal plane (from Step 1, used for fallback bounds).
    seg_area : np.ndarray
        3V segmentation mask on MSP (from Step 1).
    mcs : int
        3V centroid row on MSP (from Step 1).
    mrs : int
        3V centroid col on MSP (from Step 1).
    params : dict
        Processing parameters from ``opt_ax_plane_processing_params``.
    visualize : bool
        If True, plot WM profile and candidate planes.

    Returns
    -------
    opt_plane : int
        Optimal axial slice index for p-SA measurement.
    """
    blur_k = params["ax_plane_gauss_blur_kernel_size"]
    adapt_size = params["ax_adaptive_thresh_size"]
    if adapt_size % 2 == 0:
        adapt_size += 1
    close_k = params["lv_roi_close_kernel_size"]
    close_iter = params["num_close_iters_lv_roi_ax"]
    max_search = params["max_superior_search_ax_planes"]
    cc_col_offset = params["cc_roi_col_offset"]
    semi_maj = params["ax_lat_ven_roi_semi_major_axis"]
    semi_min = params["ax_lat_ven_roi_semi_minor_axis"]

    close_kernel = cv.getStructuringElement(
        cv.MORPH_ELLIPSE, (close_k, close_k)
    )

    # Search range: start_ax_plane going superiorly (decreasing index)
    search_planes = list(range(
        max(0, start_ax_plane - max_search),
        start_ax_plane + 1
    ))

    wm_counts = []
    plane_indices = []
    plane_debug = []  # for visualization

    n_rows, n_cols = ax_img_arr.shape[1], ax_img_arr.shape[2]
    mid_col = n_cols // 2  # approximate midline

    for pl in search_planes:
        ax_plane = ax_img_arr[pl, :, :].astype(np.float32)
        if ax_plane.max() == 0:
            continue

        # Gaussian blur
        ax_blurred = cv.GaussianBlur(ax_plane, (blur_k, blur_k), 0)
        ax_norm = cv.normalize(ax_blurred, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

        # Otsu global threshold for WM detection (replaces HU windowing)
        try:
            otsu_val = filters.threshold_otsu(ax_norm)
        except ValueError:
            continue
        ax_otsu = (ax_norm > otsu_val).astype(np.uint8)

        # Adaptive threshold to segment ventricles (local dark regions)
        ax_adapt = cv.adaptiveThreshold(
            ax_norm, 255,
            cv.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv.THRESH_BINARY_INV,
            adapt_size, 2
        )

        # Brain mask via floodfill from (0,0)
        h, w = ax_norm.shape
        flood_mask = np.zeros((h + 2, w + 2), np.uint8)
        ax_fill = ax_otsu.copy()
        res = cv.floodFill(ax_fill, flood_mask, (0, 0), 255)
        brain_mask = (1 - res[2][1:h+1, 1:w+1]).astype(np.uint8)

        # Ventricle segmentation: dark inside brain
        ven_seg = brain_mask * (1 - ax_otsu)

        # Close ventricle mask to fill gaps
        ven_closed = cv.morphologyEx(
            np.uint8(ven_seg), cv.MORPH_CLOSE, close_kernel, iterations=close_iter
        )

        # Find centroid of brain mask to anchor CC ROI
        brain_props = measure.regionprops(measure.label(brain_mask))
        if not brain_props:
            continue
        # Take largest brain region centroid
        brain_cen = max(brain_props, key=lambda r: r.area).centroid
        cen_row = int(round(brain_cen[0]))

        # CC ROI: a horizontal strip around brain centroid row, between ventricle bodies
        # Midline is approximately mid_col; CC spans ±cc_col_offset around midline
        cc_row_start = max(0, cen_row - 5)
        cc_row_end = min(n_rows, cen_row + 5)
        cc_col_start = max(0, mid_col - semi_min - cc_col_offset)
        cc_col_end = min(n_cols, mid_col + semi_min + cc_col_offset)

        cc_roi = np.zeros_like(ax_norm)
        cc_roi[cc_row_start:cc_row_end, cc_col_start:cc_col_end] = 1

        # Exclude the ventricle regions from CC ROI
        cc_roi_clean = cc_roi * (1 - np.uint8(ven_closed > 0))

        # Count WM pixels (supra-Otsu) in CC ROI
        wm_in_cc = int(np.sum(ax_otsu * cc_roi_clean))
        wm_counts.append(wm_in_cc)
        plane_indices.append(pl)
        plane_debug.append((ax_plane, ven_closed, cc_roi_clean, wm_in_cc))

    if not wm_counts:
        raise RuntimeError(
            "WM profile is empty — no valid axial slices in search range. "
            f"start_ax_plane={start_ax_plane}, max_search={max_search}."
        )

    # Optimal plane = peak WM count in CC ROI
    best_idx = int(np.argmax(wm_counts))
    opt_plane = plane_indices[best_idx]

    if visualize:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # WM profile
        axes[0].plot(plane_indices, wm_counts, "b-o", markersize=4)
        axes[0].axvline(opt_plane, color="red", linestyle="--", label=f"opt_plane={opt_plane}")
        axes[0].set_xlabel("Axial slice index")
        axes[0].set_ylabel("WM pixel count in CC ROI")
        axes[0].set_title("WM profile (Step 2)")
        axes[0].legend()

        # Optimal plane image
        best_ax, best_ven, best_cc, _ = plane_debug[best_idx]
        axes[1].set_title(f"Optimal axial plane (z={opt_plane})")
        axes[1].imshow(best_ax, cmap="gray")
        ven_vis = np.ma.masked_where(best_ven == 0, best_ven)
        axes[1].imshow(ven_vis, cmap="Blues", alpha=0.5)

        axes[2].set_title("CC ROI on optimal plane")
        axes[2].imshow(best_ax, cmap="gray")
        cc_vis = np.ma.masked_where(best_cc == 0, best_cc)
        axes[2].imshow(cc_vis, cmap="Reds", alpha=0.4)

        plt.tight_layout()
        plt.show()

    return opt_plane

## Step 3: Segment occipital horns and measure angle

On the optimal axial plane from Step 2, segment the posterior (occipital) horns of the lateral ventricles and measure the angle between their medial walls — the proxy splenial angle.

**Occipital horn isolation strategy:**
1. Segment all lateral ventricle CSF on the axial slice (Otsu + adaptive + close + floodfill)
2. Split left/right at the anatomical midline
3. For each side, isolate the posterior portion: the occipital horn is the most posterior extension. We use a posterior crop (posterior `close_roi_left_prcnt_fbc` of the bounding box) after excluding the central region near midline.
4. Trace the medial wall of each occipital horn using `trace_left_contour` / `trace_right_contour` from `mri_utils`
5. Fit OLS lines to the medial wall points using `fit_lines` → angle in degrees

**MRI adaptation:** Bilateral filter replaces CT noise pre-processing. No HU CSF window (no `csf_window_center` / `csf_window_width`). The `ven_seg_to_full_brain_ratio` threshold filters out large dark regions (e.g., sulci) that are not ventricles — this ratio is critical for MRI where sulcal CSF can be large.

In [ ]:
def segment_occipital_horns_fit_lines(ax_img_arr, opt_plane, start_ax_plane, params, visualize=False):
    """Segments occipital horns and fits lines to measure proxy splenial angle.

    Operates on the single optimal axial plane identified in Step 2.
    No HU windowing — uses Otsu + adaptive threshold directly on MRI intensities.

    Parameters
    ----------
    ax_img_arr : np.ndarray
        3D array in axial orientation, shape (Z, Y, X).
    opt_plane : int
        Optimal axial slice index from Step 2.
    start_ax_plane : int
        Starting axial slice index from Step 1 (used for fallback).
    params : dict
        Processing parameters from ``occipital_horns_segment_params``.
    visualize : bool
        If True, show segmentation and fitted lines.

    Returns
    -------
    p_sa_angle : float or None
        Proxy splenial angle in degrees, or None if segmentation fails.
    """
    from mri_utils import moore_contour, trace_left_contour, trace_right_contour

    blur_k = params["lat_ven_seg_initial_gauss_blur_size"]
    close_k = params["lv_roi_close_kernel_size"]
    close_iter = params["num_close_iters_roi_ax"]
    adapt_size = params["ax_adaptive_thresh_size"]
    if adapt_size % 2 == 0:
        adapt_size += 1
    ven_brain_ratio = params["ven_seg_to_full_brain_ratio"]
    bil_size = params["bilateral_filter_size"]
    bil_sc = params["bilater_filter_sigma_color"]
    bil_ss = params["bilater_filter_sigma_space"]
    col_offset = params["lv_middle_roi_col_offset"]
    row_offset = params["lv_middle_roi_row_offset"]
    left_prcnt = params["close_roi_left_prcnt_fbc"]
    cen_excl = params["close_roi_cen_exclude"]
    lv_close_iter = params["ax_lv_close_iter"]
    median_k = params["ven_median_smooth_factor"]
    min_area = params["min_ven_area_thresh"]

    close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_k, close_k))
    lv_close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_k, close_k))

    # ── Extract axial plane ──────────────────────────────────────────
    ax_plane = ax_img_arr[opt_plane, :, :].astype(np.float32)
    if ax_plane.max() == 0:
        print(f"Warning: opt_plane {opt_plane} is empty. Falling back to start_ax_plane {start_ax_plane}.")
        ax_plane = ax_img_arr[start_ax_plane, :, :].astype(np.float32)
        if ax_plane.max() == 0:
            return None

    n_rows, n_cols = ax_plane.shape
    mid_col = n_cols // 2

    # ── Pre-processing ───────────────────────────────────────────────
    # Bilateral filter preserves edges (ventricle walls) while smoothing interior
    ax_norm = cv.normalize(ax_plane, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
    ax_bilateral = cv.bilateralFilter(ax_norm, bil_size, bil_sc, bil_ss)

    # Gaussian blur for initial thresholding
    ax_blurred = cv.GaussianBlur(ax_bilateral.astype(np.float32), (blur_k, blur_k), 0)
    ax_blur_norm = cv.normalize(ax_blurred, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

    # ── Brain mask via Otsu + floodfill ─────────────────────────────
    try:
        otsu_val = filters.threshold_otsu(ax_blur_norm)
    except ValueError:
        return None
    ax_otsu = (ax_blur_norm > otsu_val).astype(np.uint8)

    h, w = ax_blur_norm.shape
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    ax_fill = ax_otsu.copy()
    res = cv.floodFill(ax_fill, flood_mask, (0, 0), 255)
    brain_mask = (1 - res[2][1:h+1, 1:w+1]).astype(np.uint8)
    brain_area = int(np.sum(brain_mask))
    if brain_area == 0:
        return None

    # ── Ventricle segmentation ───────────────────────────────────────
    # Adaptive threshold for local CSF detection
    ax_adapt = cv.adaptiveThreshold(
        ax_blur_norm, 255,
        cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY_INV,
        adapt_size, 2
    )

    # Ventricles = dark regions inside brain mask
    ven_raw = np.uint8(brain_mask) * np.uint8(ax_adapt > 0)

    # Close to fill gaps in ventricle contours
    ven_closed = cv.morphologyEx(ven_raw, cv.MORPH_CLOSE, close_kernel, iterations=close_iter)

    # Median filter to smooth ventricular boundaries
    ven_smooth = cv.medianBlur(np.uint8(ven_closed * 255), median_k)
    ven_smooth = (ven_smooth > 127).astype(np.uint8)

    # Remove very small components (noise) and regions too large (sulci)
    labeled_ven = measure.label(ven_smooth)
    filtered_ven = np.zeros_like(ven_smooth)
    max_ven_area = int(brain_area * ven_brain_ratio)
    for region in measure.regionprops(labeled_ven):
        if min_area <= region.area <= max_ven_area:
            filtered_ven[labeled_ven == region.label] = 1

    if filtered_ven.max() == 0:
        print(f"Warning: no ventricle regions passed area filter on plane {opt_plane}.")
        return None

    # ── Split left/right at midline ──────────────────────────────────
    left_ven = filtered_ven.copy()
    left_ven[:, mid_col:] = 0  # left hemisphere = cols < mid_col

    right_ven = filtered_ven.copy()
    right_ven[:, :mid_col] = 0  # right hemisphere = cols >= mid_col

    # ── Isolate occipital (posterior) horns ──────────────────────────
    # The occipital horn is the most posterior extension of the LV.
    # In axial view with standard orientation, posterior = larger row index.
    # We keep only the posterior fraction of each ventricle's bounding box.

    def isolate_occipital_horn(ven_half, side):
        """Keeps the posterior portion of the ventricle half after excluding central midline strip."""
        # Exclude central strip near midline
        excl_cols = int(n_cols * cen_excl)
        masked = ven_half.copy()
        if side == "left":
            masked[:, mid_col - excl_cols:] = 0
        else:
            masked[:, :mid_col + excl_cols] = 0

        if masked.max() == 0:
            return None

        # Keep only posterior fraction (large row indices)
        rows = np.where(masked > 0)[0]
        if len(rows) == 0:
            return None
        row_min, row_max = rows.min(), rows.max()
        row_range = row_max - row_min
        if row_range < 3:
            return None
        posterior_start = int(row_min + row_range * (1 - left_prcnt))
        posterior_mask = np.zeros_like(masked)
        posterior_mask[posterior_start:, :] = 1
        horn = masked * posterior_mask

        if horn.max() == 0:
            return None

        # Close the occipital horn mask
        horn_closed = cv.morphologyEx(
            np.uint8(horn), cv.MORPH_CLOSE, lv_close_kernel, iterations=lv_close_iter
        )

        # Keep largest connected component
        labeled_horn = measure.label(horn_closed)
        if labeled_horn.max() == 0:
            return None
        try:
            horn_cc = getLargestCC(labeled_horn).astype(np.uint8)
        except ValueError:
            return None

        if np.sum(horn_cc) < min_area:
            return None

        return horn_cc

    left_horn = isolate_occipital_horn(left_ven, "left")
    right_horn = isolate_occipital_horn(right_ven, "right")

    if left_horn is None or right_horn is None:
        print(f"Warning: could not isolate one or both occipital horns on plane {opt_plane}.")
        return None

    # ── Trace medial walls and fit lines ────────────────────────────
    try:
        left_contour = moore_contour(left_horn)
        right_contour = moore_contour(right_horn)
    except ValueError as e:
        print(f"Contour tracing failed: {e}")
        return None

    # Left horn: medial wall is the right (eastern) edge — highest col per row
    left_con_rows, left_con_cols = np.where(left_contour > 0)
    if len(left_con_rows) < 5:
        return None

    # Find the bottom contour point of left horn (largest row)
    bot_row_l = int(left_con_rows.max())
    bot_col_l = int(left_con_cols[left_con_rows == bot_row_l].mean())

    # Find top contour point of left horn (smallest row)
    top_row_l = int(left_con_rows.min())
    top_col_l_candidates = left_con_cols[left_con_rows == top_row_l]
    # For left horn, medial (right) edge → start from the rightmost top point
    top_col_l = int(top_col_l_candidates.max())

    left_pts, left_con_pts = trace_left_contour(top_row_l, top_col_l, bot_row_l, left_contour)

    # Right horn: medial wall is the left (western) edge — lowest col per row
    right_con_rows, right_con_cols = np.where(right_contour > 0)
    if len(right_con_rows) < 5:
        return None

    bot_row_r = int(right_con_rows.max())
    bot_col_r = int(right_con_cols[right_con_rows == bot_row_r].mean())

    top_row_r = int(right_con_rows.min())
    top_col_r_candidates = right_con_cols[right_con_rows == top_row_r]
    top_col_r = int(top_col_r_candidates.min())

    right_pts, right_con_pts = trace_right_contour(top_row_r, top_col_r, bot_row_r, right_contour)

    if len(left_con_pts) < 3 or len(right_con_pts) < 3:
        print(f"Warning: insufficient contour points for line fitting on plane {opt_plane}.")
        return None

    try:
        p_sa_angle = fit_lines(left_con_pts, right_con_pts)
    except Exception as e:
        print(f"Line fitting failed: {e}")
        return None

    if visualize:
        fig, axes = plt.subplots(1, 4, figsize=(24, 5))

        axes[0].set_title(f"Axial plane z={opt_plane}")
        axes[0].imshow(ax_plane, cmap="gray")

        axes[1].set_title("Filtered ventricle segmentation")
        axes[1].imshow(ax_plane, cmap="gray")
        ven_vis = np.ma.masked_where(filtered_ven == 0, filtered_ven)
        axes[1].imshow(ven_vis, cmap="Blues", alpha=0.5)
        axes[1].axvline(mid_col, color="yellow", linestyle="--", linewidth=1)

        axes[2].set_title("Occipital horns isolated")
        axes[2].imshow(ax_plane, cmap="gray")
        lh_vis = np.ma.masked_where(left_horn == 0, left_horn)
        rh_vis = np.ma.masked_where(right_horn == 0, right_horn)
        axes[2].imshow(lh_vis, cmap="Greens", alpha=0.6)
        axes[2].imshow(rh_vis, cmap="Reds", alpha=0.6)

        axes[3].set_title(f"Fitted lines — p-SA = {p_sa_angle:.1f}°")
        axes[3].imshow(ax_plane, cmap="gray")
        # Plot medial wall contour points
        if left_con_pts:
            lx = [pt[0] for pt in left_con_pts]
            ly = [pt[1] for pt in left_con_pts]
            axes[3].scatter(lx, ly, c="lime", s=5, zorder=3)
        if right_con_pts:
            rx = [pt[0] for pt in right_con_pts]
            ry = [pt[1] for pt in right_con_pts]
            axes[3].scatter(rx, ry, c="red", s=5, zorder=3)

        plt.suptitle(f"Step 3: Occipital horn segmentation (p-SA = {p_sa_angle:.1f}°)", fontsize=12)
        plt.tight_layout()
        plt.show()

    return p_sa_angle

## Load AC/PC coordinates

The posterior commissure (PC) x-voxel coordinate determines which sagittal slice to treat as the mid-sagittal plane in Step 1.

- **OASIS**: uses the fixed T88 atlas-space coordinate from `mri_utils.OASIS_PC_VOXEL = (85, 87, 73)`. All subjects are registered to the same space, so one coordinate works for all.
- **ABCD**: the ART pipeline outputs per-subject AC/PC `.txt` files. We load them here and build a lookup dict keyed by subject ID.
- **Custom**: implement `get_pc_voxel(scan_path)` to return `(pc_x, pc_y, pc_z)`.

In [ ]:
if ACPC_MODE == "oasis_fixed":
    # All OASIS subjects use the same T88 atlas-space coordinates
    AC_VOXEL = OASIS_AC_VOXEL  # (85, 110, 73)
    PC_VOXEL = OASIS_PC_VOXEL  # (85, 87, 73)
    print(f"Using fixed OASIS T88 atlas coordinates:")
    print(f"  AC voxel: {AC_VOXEL}")
    print(f"  PC voxel: {PC_VOXEL}")
    # Build per-subject lookup (all same)
    acpc_lookup = {}  # will default to PC_VOXEL for all subjects

elif ACPC_MODE == "abcd_txt":
    # Load per-subject AC/PC from .txt sidecar files
    acpc_lookup = {}  # subject_id -> pc_voxel tuple
    ac_lookup = {}    # subject_id -> ac_voxel tuple
    missing_acpc = []

    for _, row in scan_df.iterrows():
        subject_id = row['subject_id']
        scan_path = row['scan_path']
        # Expect sidecar at same location as scan, e.g. sub-XXX_ses-YYY_acpc.txt
        txt_path = scan_path.replace(".nii.gz", "_acpc.txt").replace(".nii", "_acpc.txt")
        if os.path.exists(txt_path):
            try:
                ac, pc = load_acpc_from_txt(txt_path)
                ac_lookup[subject_id] = tuple(ac)
                acpc_lookup[subject_id] = tuple(pc)
            except Exception as e:
                print(f"Warning: could not parse {txt_path}: {e}")
                missing_acpc.append(subject_id)
        else:
            missing_acpc.append(subject_id)

    print(f"Loaded AC/PC for {len(acpc_lookup)} subjects.")
    if missing_acpc:
        print(f"Missing AC/PC sidecar for {len(missing_acpc)} subjects: {missing_acpc[:5]}...")

elif ACPC_MODE == "custom":
    # Implement get_pc_voxel for your dataset
    def get_pc_voxel(scan_path):
        """Return (pc_x, pc_y, pc_z) voxel coordinates for a given scan path."""
        raise NotImplementedError("Implement get_pc_voxel() for your dataset.")
    acpc_lookup = {}
    print("Custom AC/PC mode: implement get_pc_voxel() above.")


def get_subject_pc_voxel(subject_id):
    """Returns PC voxel for a subject, falling back to OASIS fixed if not found."""
    if ACPC_MODE == "oasis_fixed":
        return PC_VOXEL
    return acpc_lookup.get(subject_id, OASIS_PC_VOXEL)

## Discover available scans

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path": scan_files,
})
print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Test on a single subject — all three steps with visualization

Run the full three-step pipeline on one scan to verify everything works before batch processing.

**What to expect:**

**Step 1 plots (4 panels):**
- *Mid-sagittal plane*: A left-right cross-section through the brain midline. The 3rd ventricle appears as a small dark midline slit, typically between the thalami. A cyan dashed line marks the PC row.
- *Otsu + adaptive combined*: Binary image highlighting candidate dark regions within the brain.
- *Tilted elliptical ROI*: The search region overlaid in orange/red — should encompass the 3V.
- *3V segmentation*: The identified 3V in green. The red horizontal line marks `start_ax_plane`.

**Step 2 plots (3 panels):**
- *WM profile*: Count of WM pixels in CC ROI vs axial slice index. Should show a clear peak near the CC splenium.
- *Optimal axial plane*: The axial slice at the peak, with lateral ventricles overlaid in blue.
- *CC ROI*: The CC search strip in red on the optimal plane.

**Step 3 plots (4 panels):**
- *Raw axial plane*: The optimal plane image.
- *Ventricle segmentation*: All LV CSF in blue.
- *Occipital horns isolated*: Left (green) and right (red) occipital horns.
- *Fitted lines*: Medial wall contour points and the angle measurement.

In [ ]:
# Pick first subject for testing
test_row = scan_df.iloc[0]
test_id = test_row['subject_id']
test_path = test_row['scan_path']
print(f"Testing on: {test_id}")
print(f"Path: {test_path}")

# Load volume
img = sitk.ReadImage(test_path)
print(f"Size: {img.GetSize()}, Spacing: {img.GetSpacing()}")

# Rotate to RAS for correct orientation
img_rotated = rotate_image(img)
ax_img_arr = sitk.GetArrayFromImage(img_rotated)  # shape (Z, Y, X)
print(f"Array shape (Z, Y, X): {ax_img_arr.shape}")
print(f"Intensity range: [{ax_img_arr.min()}, {ax_img_arr.max()}]")

# Get PC voxel for this subject
pc_voxel = get_subject_pc_voxel(test_id)
print(f"PC voxel: {pc_voxel}")

In [ ]:
# ── Step 1: Find starting axial plane ────────────────────────────────
print("=" * 60)
print("Step 1: Determine starting axial plane via 3V segmentation")
print("=" * 60)

start_ax_plane, msp_image, seg_area, mcs, mrs = determine_start_ax_plane(
    ax_img_arr=ax_img_arr,
    pc_voxel=pc_voxel,
    params=start_ax_plane_processing_params,
    visualize=True
)
print(f"\nstart_ax_plane = {start_ax_plane}")
print(f"3V centroid on MSP: row={mcs}, col={mrs}")

In [ ]:
# ── Step 2: Determine optimal axial plane ────────────────────────────
print("=" * 60)
print("Step 2: Determine optimal axial plane via WM profile")
print("=" * 60)

opt_plane = determine_opt_ax_plane(
    start_ax_plane=start_ax_plane,
    ax_img_arr=ax_img_arr,
    msp_image=msp_image,
    seg_area=seg_area,
    mcs=mcs,
    mrs=mrs,
    params=opt_ax_plane_processing_params,
    visualize=True
)
print(f"\nopt_plane = {opt_plane}")
print(f"Search range: [{max(0, start_ax_plane - opt_ax_plane_processing_params['max_superior_search_ax_planes'])}, {start_ax_plane}]")

In [ ]:
# ── Step 3: Segment occipital horns and measure angle ────────────────
print("=" * 60)
print("Step 3: Segment occipital horns and measure p-SA")
print("=" * 60)

p_sa_angle = segment_occipital_horns_fit_lines(
    ax_img_arr=ax_img_arr,
    opt_plane=opt_plane,
    start_ax_plane=start_ax_plane,
    params=occipital_horns_segment_params,
    visualize=True
)

if p_sa_angle is not None:
    print(f"\np-SA for {test_id}: {p_sa_angle:.2f}°")
else:
    print(f"\np-SA computation failed for {test_id}. Inspect visualization above.")

## Batch processing — all subjects

Run the full three-step pipeline on all subjects with visualization off. Results include per-subject plane indices for QC.

In [ ]:
results = []

for _, row in scan_df.iterrows():
    subject_id = row['subject_id']
    vol_path = row['scan_path']

    try:
        # Load and orient volume
        img = sitk.ReadImage(vol_path)
        ax_img_arr_batch = sitk.GetArrayFromImage(rotate_image(img))

        # Get PC voxel for this subject
        pc_vox = get_subject_pc_voxel(subject_id)

        # Step 1
        sap, msp_img, seg_a, mcs_b, mrs_b = determine_start_ax_plane(
            ax_img_arr=ax_img_arr_batch,
            pc_voxel=pc_vox,
            params=start_ax_plane_processing_params,
            visualize=False
        )

        # Step 2
        opt_p = determine_opt_ax_plane(
            start_ax_plane=sap,
            ax_img_arr=ax_img_arr_batch,
            msp_image=msp_img,
            seg_area=seg_a,
            mcs=mcs_b,
            mrs=mrs_b,
            params=opt_ax_plane_processing_params,
            visualize=False
        )

        # Step 3
        angle = segment_occipital_horns_fit_lines(
            ax_img_arr=ax_img_arr_batch,
            opt_plane=opt_p,
            start_ax_plane=sap,
            params=occipital_horns_segment_params,
            visualize=False
        )

        results.append({
            "subject_id": subject_id,
            "p_sa_angle": angle,
            "start_ax_plane": sap,
            "opt_ax_plane": opt_p,
        })

        status = f"{angle:.2f}°" if angle is not None else "FAILED"
        print(f"{subject_id}: p-SA={status} (start={sap}, opt={opt_p})")

    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        results.append({
            "subject_id": subject_id,
            "p_sa_angle": np.nan,
            "start_ax_plane": np.nan,
            "opt_ax_plane": np.nan,
        })

results_df = pd.DataFrame(results)
n_valid = results_df['p_sa_angle'].notna().sum()
print(f"\nProcessed {len(results_df)} subjects, {n_valid} valid p-SA measurements.")

## Results — distribution and CSV export

**What to expect:**

**p-SA distribution histogram:** In healthy young adults (OASIS CDR=0), p-SA is typically in the range 80–110°. Smaller angles indicate more acute occipital horn divergence, which can be associated with ventriculomegaly or posterior fossa abnormalities.

**By CDR group (if demographics available):** Subjects with higher CDR scores (more dementia) tend to have wider angles due to ventricular enlargement expanding the occipital horns laterally — widening the angle between the medial walls.

In [ ]:
# ── Load demographics (optional) ────────────────────────────────────
demo_df = None
if DEMOGRAPHICS_PATH and os.path.exists(DEMOGRAPHICS_PATH):
    ext = os.path.splitext(DEMOGRAPHICS_PATH)[1].lower()
    if ext in (".xlsx", ".xls"):
        demo_df = pd.read_excel(DEMOGRAPHICS_PATH)
    elif ext == ".csv":
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH)
    elif ext in (".tsv", ".txt"):
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH, sep="\t")
    if demo_df is not None:
        print(f"Loaded demographics: {len(demo_df)} subjects")
        # Merge with results
        id_col = next((c for c in demo_df.columns if c.upper() in ("ID", "SUBJECT_ID")), None)
        if id_col:
            keep = [id_col] + [c for c in demo_df.columns
                               if c in ('Age', 'M/F', 'CDR', 'Sex', 'age', 'sex')]
            results_df = results_df.merge(demo_df[keep], left_on='subject_id', right_on=id_col, how='left')

# ── Plot ─────────────────────────────────────────────────────────────
has_cdr = 'CDR' in results_df.columns and results_df['CDR'].notna().any()
n_plots = 2 if has_cdr else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

# Histogram of p-SA
valid_angles = results_df['p_sa_angle'].dropna()
axes[0].hist(valid_angles, bins=20, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Proxy Splenial Angle (degrees)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'p-SA Distribution (n={len(valid_angles)})')
if len(valid_angles) > 0:
    axes[0].axvline(valid_angles.mean(), color='red', linestyle='--',
                    label=f'Mean = {valid_angles.mean():.1f}°')
    axes[0].legend()

# By CDR group
if has_cdr:
    for cdr_val in sorted(results_df['CDR'].dropna().unique()):
        subset = results_df[results_df['CDR'] == cdr_val]['p_sa_angle'].dropna()
        if len(subset) > 0:
            axes[1].hist(subset, bins=15, alpha=0.55, label=f'CDR {cdr_val} (n={len(subset)})')
    axes[1].set_xlabel('Proxy Splenial Angle (degrees)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('p-SA by Dementia Severity (CDR)')
    axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\np-SA summary (n={len(valid_angles)}):")
print(results_df['p_sa_angle'].describe().round(2))
if has_cdr:
    print("\np-SA by CDR group:")
    print(results_df.groupby('CDR')['p_sa_angle'].describe().round(2))

# QC: plane selection summary
print("\nPlane selection summary:")
print(results_df[['start_ax_plane', 'opt_ax_plane']].describe().round(1))

In [ ]:
output_path = os.path.join(OUTPUT_DIR, f"proxy_splenial_angle_{DATASET}.csv")
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"Columns: {list(results_df.columns)}")
results_df.head(10)